# ML-03 - Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sinagdevv/flyrank-ml-internship-sinagdev/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This frame keeps Lane 2 focused on an interpretable review-priority decision. The measurements below use only the authorized starter CSV.

## 1. My lane as an ML task (type)

This is Lane 2: Refresh / Content Opportunity Scoring. The ML task is page-level ranking (a scoring problem): produce an interpretable priority score so an SEO editor or content strategist can review the highest-priority pages first. The resulting action is human review, not an automatic refresh. A transparent stale-and-visible rule is the baseline. A learned score is only a hypothesis for combining several observable signals more usefully; it has not earned that claim until client-grouped validation shows an improvement in Precision@50.

In [1]:
candidate_features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate", "content_type"]
forbidden_features = {"content_id", "client_id", "trend_direction", "trend_pct"}
assert not forbidden_features.intersection(candidate_features)
print("Task type: interpretable page-level ranking")
print("Candidate features:", candidate_features)
print("Forbidden identifiers/outcome-derived fields stay out of features:", sorted(forbidden_features))

Task type: interpretable page-level ranking
Candidate features: ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate', 'content_type']
Forbidden identifiers/outcome-derived fields stay out of features: ['client_id', 'content_id', 'trend_direction', 'trend_pct']


## 2. Target or proxy

The provisional proxy is `observed_decline = (trend_direction == 'down')`. This is an observed current 90-day directional outcome in the starter snapshot, not a future label and not a claim that a refresh will cause recovery. `trend_direction` is derived from `trend_pct`; both fields define the outcome and are explicitly excluded from predictive features. The provisional visibility cutoff is `impressions_90d >= 500`; it scopes the review inventory and is not optimized.

In [2]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "data/raw/content_refresh_anonymized.csv").exists():
    repo_root = (repo_root / "../..").resolve()
data_path = repo_root / "data/raw/content_refresh_anonymized.csv"
assert data_path.exists(), f"Starter CSV not found: {data_path}"
df = pd.read_csv(data_path)
required = {"content_id", "client_id", "impressions_90d", "days_since_last_update", "trend_direction", "trend_pct"}
assert required <= set(df.columns)
assert df[list(required - {"trend_pct"})].notna().all().all()
df["observed_decline"] = df["trend_direction"].astype("string").str.lower().eq("down")
print(f"Loaded {len(df):,} rows and {df['client_id'].nunique():,} clients from the authorized starter CSV.")
print(f"Observed current down cases: {int(df['observed_decline'].sum()):,} ({df['observed_decline'].mean():.1%})")
print("Target source: trend_direction == 'down'; trend_direction and trend_pct are not predictive features.")

Loaded 30,000 rows and 32 clients from the authorized starter CSV.
Observed current down cases: 16,262 (54.2%)
Target source: trend_direction == 'down'; trend_direction and trend_pct are not predictive features.


## 3. Success metric

The primary metric is Precision@50: among the 50 pages at the top of the scored review queue, the fraction with the observed current decline proxy. Higher is better because editor review capacity is limited. Compare the learned ranking with the transparent stale-and-visible baseline using the same client-grouped holdout. The comparison is decision support, not a causal estimate.

In [3]:
visibility_cutoff = 500  # provisional editorial cutoff, not optimized
eligible = df.loc[df["impressions_90d"].ge(visibility_cutoff)].copy()
assert len(eligible) > 50

def precision_at_50(frame, score_column):
    top = frame.nlargest(50, score_column)
    return float(top["observed_decline"].mean())

print(f"Visible review inventory (impressions_90d >= {visibility_cutoff:,}): {len(eligible):,} pages")
print("Metric: Precision@50 = observed_decline rate among the top 50 scored pages.")
print("Validation: client-grouped holdout; no client ID is used as a feature.")

Visible review inventory (impressions_90d >= 500): 16,726 pages
Metric: Precision@50 = observed_decline rate among the top 50 scored pages.
Validation: client-grouped holdout; no client ID is used as a feature.


## 4. The unit of analysis, as a real dataframe

The unit is one pseudonymized content page. The review slice keeps one row per page with `content_id` retained only to demonstrate grain, and `client_id` retained only for grouped validation. The current decline proxy is shown for measurement, but outcome-defining fields remain excluded from any future score.

In [4]:
page_columns = ["content_id", "client_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate", "content_type", "observed_decline"]
page_frame = eligible[page_columns].copy()
assert page_frame["content_id"].is_unique
assert len(page_frame) == page_frame["content_id"].nunique()
print(f"Dataframe shape: {page_frame.shape}")
print(f"Unique pages: {page_frame['content_id'].nunique():,}; unique clients: {page_frame['client_id'].nunique():,}")
print(f"Provisional stale-visible pages (days_since_last_update >= 180): {int(page_frame['days_since_last_update'].ge(180).sum())}")
display(page_frame.head())

Dataframe shape: (16726, 9)
Unique pages: 16,726; unique clients: 28
Provisional stale-visible pages (days_since_last_update >= 180): 17


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,engagement_rate,content_type,observed_decline
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,5.88,keyword article,True
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,0.00,keyword article,True
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,0.00,keyword article,True
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,1.28,keyword article,False
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,0.00,keyword article,True


## 5. Why ML beats a fixed rule here

The fixed baseline can flag pages that are both visible and stale. A learned ranking could outperform it if interactions among exposure, freshness, position, CTR, engagement, and content type identify current decline cases that the two-condition rule misses. That is a testable hypothesis, not an established result. The comparison must use the same Precision@50 metric and client-grouped holdout, prevent client leakage, and keep human review required. Current decline is not a future outcome, the observed association is not causal, and the visibility and staleness cutoffs remain provisional rather than optimized.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

forbidden = {"content_id", "client_id", "trend_direction", "trend_pct", "observed_decline"}
assert not forbidden.intersection(candidate_features)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, holdout_idx = next(splitter.split(page_frame, groups=page_frame["client_id"]))
train_clients = set(page_frame.iloc[train_idx]["client_id"])
holdout_clients = set(page_frame.iloc[holdout_idx]["client_id"])
assert train_clients.isdisjoint(holdout_clients)
print(f"Candidate feature count: {len(candidate_features)}; excluded outcome/ID fields: {len(forbidden)}")
print(f"Client-grouped split: {len(train_idx):,} train pages / {len(holdout_idx):,} holdout pages; client overlap: {len(train_clients & holdout_clients)}")

Candidate feature count: 6; excluded outcome/ID fields: 5
Client-grouped split: 15,466 train pages / 1,260 holdout pages; client overlap: 0


## Self-check

- [x] Lane, decision-maker, action, task type, target proxy, metric, and validation are explicit.
- [x] The notebook shows a real one-row-per-page dataframe from the authorized starter CSV.
- [x] `trend_direction` and `trend_pct` are excluded from predictive features when defining the outcome.
- [x] Current decline is described as observed and directional, not future or causal.
- [x] Visibility and staleness cutoffs are labeled provisional; human review remains required.
- [ ] Commit and submit to the internship portal (intentionally not performed in this task).